# Transformer (RUL FD001)

Busca Hyperband, treino repetido e gráficos do Transformer encoder. A tabela de iterações fica em `data/processed/results/transformer_results.csv` para `src/comparacao_wilcoxon.ipynb`.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

NOTEBOOK_DIR = Path.cwd()
if (NOTEBOOK_DIR / "data.py").is_file():
    PROJECT_ROOT = NOTEBOOK_DIR.parent
else:
    PROJECT_ROOT = NOTEBOOK_DIR

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data import (
    CYCLE_COLUMN,
    DEFAULT_RUL_LIMIT,
    ID_COLUMN,
    assign_fd001_column_names,
    compute_test_rul,
    compute_train_rul,
    drop_missing_rows,
    drop_unused_sensor_columns,
    load_fd001_raw,
)
from src.features import create_fd001_windowed_features
from src.models import (
    plot_rul_prediction_diagnostics,
    plot_training_history,
    predict_rul,
    run_repeated_train_eval,
    save_best_params,
    save_experiment_results,
    search_hyperparameters,
    show_figure,
    split_windowed_data,
)

DATA_RAW_DIR = PROJECT_ROOT / "data" / "raw"
TUNER_DIR = PROJECT_ROOT / "data" / "processed" / "keras_tuner"
FIGURES_DIR = PROJECT_ROOT / "data" / "processed" / "figures"
RESULTS_DIR = PROJECT_ROOT / "data" / "processed" / "results"
TUNER_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

RUL_LIMIT = DEFAULT_RUL_LIMIT
WINDOW_SIZE = 30
WINDOW_STEP = 1
N_ITER = 10

CHECKPOINT = PROJECT_ROOT / "data" / "processed" / "transformer_model.h5"
RESULTS_PATH = RESULTS_DIR / "transformer_results.csv"
PARAMS_PATH = RESULTS_DIR / "transformer_best_params.json"


## Carga e exploração inicial

In [ ]:
raw = load_fd001_raw(DATA_RAW_DIR)
train_df, test_df = drop_missing_rows(raw.train, raw.test)
train_df, test_df, rul_df = assign_fd001_column_names(train_df, test_df, raw.rul)

print(train_df.head())

In [ ]:
fig, axes = plt.subplots(nrows=6, ncols=4, figsize=(24, 20))
axes = axes.ravel()
for i, item in enumerate(train_df.columns[2:]):
    train_df.groupby(ID_COLUMN).plot(
        kind="line", x=CYCLE_COLUMN, y=item, ax=axes[i]
    )
    axes[i].set_title(f"{item}")
    axes[i].get_legend().remove()
    axes[i].set_xlabel("Número de ciclos")
    axes[i].set_ylabel("")

plt.subplots_adjust(top=0.99, bottom=0.01, hspace=0.3, wspace=0.2)
show_figure(fig, FIGURES_DIR / "sensors.png")

## Features, targets e janelas temporais

In [ ]:
train_model, test_model = drop_unused_sensor_columns(train_df, test_df)
rul_train = compute_train_rul(train_model, limit=RUL_LIMIT)
rul_test = compute_test_rul(test_model, rul_df, limit=RUL_LIMIT)

windowed = create_fd001_windowed_features(
    train_model,
    test_model,
    rul_train,
    rul_test,
    window_size=WINDOW_SIZE,
    step=WINDOW_STEP,
)
x_train, y_train = windowed.x_train, windowed.y_train
x_test, y_test = windowed.x_test, windowed.y_test

print(x_train.shape, y_train.shape)
print(x_test.shape, y_test.shape)

In [ ]:
exemplo = train_df[train_df[ID_COLUMN] == 1]
n_cycles = len(exemplo[CYCLE_COLUMN])
rul_motor1 = compute_train_rul(exemplo, limit=RUL_LIMIT)

plt.figure(figsize=(10, 6))
plt.plot(exemplo[CYCLE_COLUMN].iloc[:n_cycles], rul_motor1[:n_cycles], c="red")
plt.title("Representação do ciclo de vida do motor ID = 1")
plt.ylim(0, 140)
plt.xlim(0, 200)
plt.xlabel("Ciclo do motor")
plt.ylabel("RUL")
show_figure(plt.gcf(), FIGURES_DIR / "rul_engine_1.png")

## Partição treino/validação interna

In [ ]:
x1, x2, y1, y2 = split_windowed_data(x_train, y_train, test_size=0.2, random_state=1)

## Transformer: busca de hiperparâmetros e avaliação repetida

In [ ]:
best_model, best_params = search_hyperparameters(
    "transformer",
    x1,
    y1,
    x2,
    y2,
    tuner_directory=TUNER_DIR,
    project_name="hyper_transformer_hyperband",
    max_epochs=15,
    factor=3,
    hyperband_iterations=1,
    batch_size=200,
    patience=3,
    overwrite=False,
)

save_best_params(best_params, PARAMS_PATH)
for key, value in best_params.items():
    print(f"{key:15s} -> {value}")

In [ ]:
resultados, history = run_repeated_train_eval(
    best_model,
    x1,
    y1,
    x2,
    y2,
    x_test,
    y_test,
    CHECKPOINT,
    n_iterations=N_ITER,
    capture_first_history=True,
)
save_experiment_results(resultados, RESULTS_PATH)
print(resultados.to_string(index=False))

if history is not None:
    plot_training_history(
        history,
        "Transformer",
        save_path=FIGURES_DIR / "transformer_training_history.png",
    )
    y_pred = predict_rul(best_model, x_test)
    plot_rul_prediction_diagnostics(
        y_test,
        y_pred,
        model_label="Transformer",
        rul_limit=RUL_LIMIT,
        figure_dir=FIGURES_DIR,
    )